This notebook is about getting start with genAI. Steps followed in this notebook:
1) Downloaded the below-listed models
2) Using Qwen2.5 (3 billions of parameters) using quantization so that it can run on a 4GB GPU
3) Defining a KB (facts). using BGR base 1.5 to get embedding for the facts and ChromaDB to build a DB and for VectorDB use.
4) Using RAG to answer questions using the same KB and tools used in previous point

In [1]:
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction


/home/davide/miniconda3/envs/qwen/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# QWEN 2.5

In [2]:
# my model path
model_path = "/media/davide/DATA/qwen/qwen2_5-3B/"

# define quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

# instantiate tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="cuda",
)
model.eval()


Loading weights: 100%|██████████| 434/434 [00:05<00:00, 77.76it/s] 


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm

In [16]:
messages = []

i=0

while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        break

    i+=1
    print(i,"You:",user_input)
    messages.append({"role": "user", "content": user_input})


    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    generated_ids = model.generate(**model_inputs, max_new_tokens=512)

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    messages.append({"role": "assistant", "content": response})
    print("Qwen:", response, "\n\n\n")

1 You: hi, introduce yourself 


Qwen: Hello! I'm Qwen, a large language model created by Alibaba Cloud. My primary function is to assist with various tasks and provide information across a wide range of topics. How can I help you today? 



2 You: can u tell me a bit more about your achitecture?

Qwen: Certainly! As a large language model, my architecture is quite complex and involves several key components designed to enable me to process and generate text effectively. Here's an overview of the main aspects that make up my architecture:

### 1. **Neural Network Architecture**
   - **Transformer Encoder-Decoder**: This is a core component that allows me to handle sequence-to-sequence tasks like translation and text generation. The Transformer model uses self-attention mechanisms instead of traditional recurrent neural networks (RNNs) or convolutional neural networks (CNNs), which helps in capturing long-range dependencies in the input sequence.

### 2. **Pre-training**
   - **Masked L

# Vector DB

In [3]:
#knowledge base
scientific_facts = [
    "Water reaches its triple point at exactly 0.01°C and 611.657 pascals of pressure, where it can exist as solid, liquid, and gas all at the same time. This unique balance is so precise that it's used to define the Kelvin temperature scale.",
    "Bananas are mildly radioactive because they contain potassium-40, a naturally occurring isotope found in many foods. You'd need to eat roughly 10 million bananas at once to receive a lethal dose of radiation, so they're perfectly safe.",
    "A single day on Venus lasts about 243 Earth days, which is actually longer than the 225 Earth days it takes the planet to complete one orbit around the Sun. In other words, a Venusian day is longer than a Venusian year.",
    "Octopuses have three hearts: two pump blood to the gills while the third pumps it to the rest of the body. Their blood is blue because it uses copper-based hemocyanin instead of iron-based hemoglobin to carry oxygen.",
    "A lightning bolt can heat the air around it to roughly 30,000 Kelvin, which is about five times hotter than the surface of the Sun. This extreme heat causes the air to expand explosively, producing the thunder we hear.",
    "The human body actually emits a faint visible glow through biophotons, but the light is about 1,000 times too weak for the naked eye to detect. Scientists can only observe it using highly sensitive cameras in complete darkness.",
    "The Eiffel Tower can grow up to 15 centimeters taller during hot summer days because the iron expands as it heats up. The tower subtly shrinks back down when temperatures cool, making it a giant thermometer of sorts.",
    "Sound travels through water at roughly 1,480 meters per second, which is more than four times faster than its speed of about 343 meters per second in air. This is why underwater sounds can travel vast distances and why whales can communicate across oceans.",
    "Earth is home to an estimated 3 trillion trees, which is far more than the 100 to 400 billion stars in the Milky Way galaxy. Even so, humans are cutting down around 15 billion trees every year.",
    "Under certain conditions, hot water can freeze faster than cold water, a counterintuitive phenomenon known as the Mpemba effect. Scientists still debate the exact cause, but factors like evaporation, convection, and dissolved gases may all play a role.",
    "The average human body contains roughly 37.2 trillion cells, according to a 2013 estimate that revised earlier figures. These cells work together in complex systems that keep us alive, from red blood cells carrying oxygen to neurons firing thoughts.",
    "Glass is not a slow-flowing liquid at room temperature, despite the popular myth that old windows are thicker at the bottom because the glass sagged over time. In reality, antique windows simply have uneven thickness because of how they were manufactured.",
    "A single teaspoon of neutron star material would weigh about 6 billion tons on Earth, since neutron stars are incredibly dense. Just a sugar-cube-sized amount contains more mass than Mount Everest.",
    "Bacteria in and on the human body outnumber human cells, though the ratio is closer to 1.3 bacteria per human cell rather than the often-cited 10 to 1. Most of these microbes are harmless or even helpful, aiding digestion and supporting the immune system.",
    "The Moon is slowly drifting away from Earth at a rate of about 3.8 centimeters per year, caused by tidal interactions between the two bodies. Over millions of years, this means total solar eclipses will eventually stop happening.",
    "Helium remains a liquid even at absolute zero under normal pressure, which is unique among all elements. It only becomes a solid when subjected to about 25 atmospheres of pressure.",
    "Human stomach acid has a pH between 1.5 and 3.5, making it strong enough to dissolve metals like zinc and even corrode some coins. The stomach protects itself with a thick layer of mucus that is constantly renewed.",
    "Sharks have been swimming in Earth's oceans for around 450 million years, which means they existed long before trees appeared roughly 350 million years ago. They've survived multiple mass extinctions and predate even the dinosaurs.",
    "A photon created in the Sun's core takes about 100,000 years to fight its way to the surface, but only about 8 minutes to travel from there to Earth. This means the sunlight you feel today began its journey long before humans existed.",
    "Scientists have successfully stored digital data in synthetic DNA, including entire books, images, and even a short GIF. DNA is incredibly dense and durable, potentially capable of preserving information for thousands of years.",
]

humanity_facts = [
    "Cleopatra lived closer in time to the first Moon landing than to the construction of the Great Pyramid of Giza. The pyramid was built around 2560 BCE, Cleopatra ruled around 30 BCE, and humans landed on the Moon in 1969 CE.",
    "Teaching at Oxford University began around 1096 CE, making it older than the Aztec Empire, which founded its capital Tenochtitlan around 1325 CE. Oxford has been continuously operating for over 900 years.",
    "The Anglo-Zanzibar War of 1896 holds the record for the shortest war in history, lasting only about 38 minutes. The conflict ended so quickly that the entire fighting force on one side was essentially neutralized before lunch.",
    "Nintendo was founded in 1889 as a company selling handmade playing cards in Japan, long before it became a video game giant. It took nearly a century for the company to shift into the electronic entertainment industry.",
    "The Hundred Years' War between England and France actually lasted 116 years, from 1337 to 1453. The name is a historical convenience rather than an accurate description of its length.",
    "Ancient Romans used urine as a mouthwash because the ammonia in it acted as a cleaning and whitening agent. Some Romans even collected urine to sell, and the emperor Vespasian famously taxed it.",
    "The Great Wall of China is often said to be visible from space with the naked eye, but this is a myth. Astronauts have confirmed that it's too narrow and blends too much with the surrounding landscape to be seen without aid.",
    "Written language emerged around 5,200 years ago with Sumerian cuneiform in Mesopotamia, one of the earliest known writing systems. It began as a way to record trade and gradually evolved into a full script.",
    "The Ottoman Empire lasted an impressive 623 years, from roughly 1299 to 1922. At its height, it spanned three continents and controlled vast trade routes between Europe and Asia.",
    "Mandarin Chinese has the most native speakers of any language in the world, with around 918 million people speaking it as their first language. It's followed by Spanish and English in terms of native speakers.",
    "The first recorded Olympic Games took place in 776 BCE in Olympia, Greece, and were held every four years for nearly 12 centuries. They were eventually banned in 393 CE before being revived in 1896.",
    "Baghdad was likely the largest city in the world around 900 CE, with a population of roughly one million people. It was a center of learning, trade, and culture during the Islamic Golden Age.",
    "Johannes Gutenberg invented the printing press around 1440, revolutionizing the spread of knowledge across Europe. His innovations made books affordable and helped fuel the Renaissance and Reformation.",
    "Women in Switzerland didn't gain the right to vote at the federal level until 1971, making it one of the last Western democracies to grant suffrage. Some Swiss cantons held out even longer at the local level.",
    "The Berlin Wall fell on November 9, 1989, after 28 years of dividing East and West Berlin. Its collapse became a powerful symbol of the end of the Cold War.",
    "The Indus Valley Civilization had advanced indoor plumbing and sewage systems as early as 2600 BCE, centuries before many later civilizations developed similar technology. Cities like Mohenjo-daro had private bathrooms and covered drains.",
    "Napoleon Bonaparte was actually around 5 feet 7 inches tall, which was average for a Frenchman of his time. The myth of him being short came largely from British propaganda and confusion over French versus English measurement units.",
    "The Aztec Empire required all children to attend school, regardless of their social class or gender. Boys and girls learned different skills, but education was considered essential for everyone.",
    "Ada Lovelace is widely regarded as the first computer programmer for her work on Charles Babbage's Analytical Engine in the 1840s. She wrote an algorithm intended for the machine, decades before computers were actually built.",
    "Humans have been creating art for at least 40,000 years, as shown by cave paintings like those in Chauvet, France. These ancient artworks depict animals, hunting scenes, and abstract symbols, revealing that creativity is deeply rooted in human nature.",
]

In [4]:
kb = scientific_facts+humanity_facts
print(len(scientific_facts), len(humanity_facts), len(kb))

20 20 40


In [5]:
# load DB
chroma_path = "/media/davide/DATA/qwen/chroma_db"
client = chromadb.PersistentClient(path=chroma_path)

# load embedder
embedding_function = SentenceTransformerEmbeddingFunction(
    "/media/davide/DATA/qwen/bge-base-en1.5",
    device="cuda"
)

collection = client.get_or_create_collection(
    name="my_knowledge_base",
    embedding_function=embedding_function,
    metadata={"hnsw:space": "cosine"}
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4206.73it/s]


In [6]:
 # populate DB
ids = [f"fact_{i}" for i in range(len(kb))]

collection.upsert(
    documents=kb,
    ids=ids
)
print(f"Collection now contains {collection.count()} documents.")

Collection now contains 40 documents.


In [7]:
# 6. Query the collection function
def query_chroma(query_text, n_results=3):
    results = collection.query(
        query_texts=[query_text],
        n_results=n_results
    )
    return results

In [8]:
#'--- Test the query ---
for query in ["Who was Cleopatra?", "How is living in Venus like?"]:
    results = query_chroma(query)

    print(f"\nQuery: '{query}'")
    print("Top matches:")

    # The results object contains 'documents', 'ids', and 'distances'
    for i, doc in enumerate(results['documents'][0]):
        distance = results['distances'][0][i]
        print(f"  Distance: {distance:.4f}  |  {doc}")


Query: 'Who was Cleopatra?'
Top matches:
  Distance: 0.6319  |  Cleopatra lived closer in time to the first Moon landing than to the construction of the Great Pyramid of Giza. The pyramid was built around 2560 BCE, Cleopatra ruled around 30 BCE, and humans landed on the Moon in 1969 CE.
  Distance: 0.8973  |  Baghdad was likely the largest city in the world around 900 CE, with a population of roughly one million people. It was a center of learning, trade, and culture during the Islamic Golden Age.
  Distance: 0.9866  |  Ada Lovelace is widely regarded as the first computer programmer for her work on Charles Babbage's Analytical Engine in the 1840s. She wrote an algorithm intended for the machine, decades before computers were actually built.

Query: 'How is living in Venus like?'
Top matches:
  Distance: 0.7238  |  A single day on Venus lasts about 243 Earth days, which is actually longer than the 225 Earth days it takes the planet to complete one orbit around the Sun. In other words,

# RAG

In [11]:
def ask_with_rag(user_query, k=3):
    # 1. Retrieve relevant facts
    results = query_chroma(user_query, n_results=k)
    retrieved_docs = results['documents'][0]

    # 2. Build context string
    context = "\n".join([f"- {doc}" for doc in retrieved_docs])

    # 3. Build system prompt with context
    system_prompt = (
        "You are a helpful assistant. Use only the context below to answer the user's question. "
        "If the answer is not in the context, say so clearly. Do not use prior knowledge.\n\n"
        f"Context:\n{context}"
    )

    # 4. Build messages
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query},
    ]

    # 5. Generate
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    generated_ids = model.generate(**model_inputs, max_new_tokens=512)

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response, retrieved_docs

In [15]:

for questions in ["How is Venus like?" ,"How long are days and years in Venus"]:
    response, documents = ask_with_rag(questions, k=2)
    print(f"Question: {questions}")
    print(f"Answer: {response}")
    print("Documents:")
    for doc in documents:
        print(doc)
    print("\n\n")

Question: How is Venus like?
Answer: The information provided does not directly compare Venus to any other body or concept. It describes characteristics of Venus such as its extremely long day and year, and the slow travel time of sunlight from the Sun to Earth. Without additional context, I cannot definitively state how Venus is "like" something else based solely on this information.
Documents:
A single day on Venus lasts about 243 Earth days, which is actually longer than the 225 Earth days it takes the planet to complete one orbit around the Sun. In other words, a Venusian day is longer than a Venusian year.
A photon created in the Sun's core takes about 100,000 years to fight its way to the surface, but only about 8 minutes to travel from there to Earth. This means the sunlight you feel today began its journey long before humans existed.



Question: How long are days and years in Venus
Answer: A day on Venus lasts about 243 Earth days, while a year on Venus lasts about 225 Earth d